# DiffPrep + H2O AutoML

This notebook runs the original DiffPrep pipeline on the exact 30-dataset
ACORec test suite, then sends its transformed data to H2O. The separate
H2O-baselines notebook owns No Preprocessing and H2O Default results.

H2O target encoding is disabled. H2O still performs its native categorical
and missing-value handling inside individual models.
The model is selected on validation and scored once on the outer test.
Run five Save-Version jobs with `DATASET_SHARD_INDEX=0..4`.


In [ ]:
# Install DiffPrep dependencies and H2O; TPOT is not used here.
%pip install -q "h2o==3.46.0.11" "impyute>=0.0.8" "pyarrow>=15" "requests"


In [ ]:
from __future__ import annotations
import gc
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
import traceback
import warnings
from pathlib import Path

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import torch
import h2o
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

RUN_MODE = "smoke"       # change to final after the one-dataset smoke run
NUM_DATASET_SHARDS = 5
DATASET_SHARD_INDEX = 0
METHOD = "diffprep_fix"
SPLIT_SEED = 42
TRAIN_SEED = 1
MAX_SAMPLES = 100_000
PARQUET_BATCH_SIZE = 4_096
H2O_MAX_RUNTIME_SECS = 120 if RUN_MODE == "smoke" else 300
H2O_MAX_RUNTIME_SECS_PER_MODEL = 60
H2O_NFOLDS = 5
H2O_NTHREADS = 1
H2O_MAX_MEM_SIZE = "6G"

KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = Path("/kaggle/working/diffprep_h2o") if KAGGLE else Path("outputs/diffprep_h2o")
TEMP_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else OUTPUT_DIR / "temp"
REPO_DIR = TEMP_ROOT / "DiffPrep"
SOLUTION_DIR = TEMP_ROOT / "SolutionRecommendation"
CACHE_DIR = TEMP_ROOT / "openml_datagit_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
if RUN_MODE not in {"smoke", "final"} or not 0 <= DATASET_SHARD_INDEX < NUM_DATASET_SHARDS:
    raise ValueError("DATASET_SHARD_INDEX out of range")
print("H2O:", h2o.__version__)


In [ ]:
# Clone the exact DiffPrep fork and the repository containing the shared
# H2O evaluator utility.
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "kaggle-experiments", "--single-branch", "https://github.com/dangvu53/DiffPrep.git", str(REPO_DIR)], check=True)
else:
    # A previous run may have left the leakage patch (or a failed patch)
    # in this disposable Kaggle checkout. Restore the exact fork before
    # applying the current patch below.
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "kaggle-experiments"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", "kaggle-experiments"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/kaggle-experiments"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if (SOLUTION_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(SOLUTION_DIR), "fetch", "origin", "feature/acorec-autodp-space"], check=True)
    subprocess.run(["git", "-C", str(SOLUTION_DIR), "switch", "feature/acorec-autodp-space"], check=True)
    subprocess.run(["git", "-C", str(SOLUTION_DIR), "pull", "--ff-only", "origin", "feature/acorec-autodp-space"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "feature/acorec-autodp-space", "--single-branch", "https://github.com/MothMalone/SolutionRecommendation.git", str(SOLUTION_DIR)], check=True)
solution_commit = subprocess.check_output(["git", "-C", str(SOLUTION_DIR), "rev-parse", "HEAD"], text=True).strip()
# Guard against an old notebook/session cache before a long run.
evaluator_path = SOLUTION_DIR / "scripts" / "h2o_evaluator.py"
evaluator_source = evaluator_path.read_text(encoding="utf-8")
required_evaluator_markers = (
    "def _classification_labels",
    "test_prediction = _classification_labels(test_prediction)",
)
if (
    "h2o.init(nthreads=int(nthreads), max_mem_size=str(max_mem_size), silent=True)" in evaluator_source
    or not all(marker in evaluator_source for marker in required_evaluator_markers)
):
    raise RuntimeError(
        "Stale H2O evaluator detected. Restart the Kaggle session and rerun this clone/install cell."
    )
sys.path.insert(0, str(SOLUTION_DIR / "scripts"))
sys.path.insert(0, str(SOLUTION_DIR / "src"))
import importlib
importlib.invalidate_caches()
from h2o_evaluator import evaluate_h2o_frames
from automl_aco.data.loaders import load_gitlab_openml_dataset
from automl_aco.eval_ids import EVAL_IDS

# The upstream trainer evaluates X_test every epoch and passes it into
# pipeline initialization. Patch that behavior in this reproduction:
# DiffPrep may use train/validation only; outer test is reserved for H2O.
def patch_exact(path, old, new):
    path = Path(path)
    source = path.read_text(encoding="utf-8")
    if old in source:
        path.write_text(source.replace(old, new), encoding="utf-8")
    elif new not in source:
        raise RuntimeError(f"DiffPrep leakage patch target not found: {path}")

for pipeline_name in ("diffprep_fix_pipeline.py", "diffprep_flex_pipeline.py"):
    pipeline_path = REPO_DIR / "pipeline" / pipeline_name
    patch_exact(
        pipeline_path,
        "return df.isnull().values.sum() > 0",
        "return df is not None and df.isnull().values.sum() > 0",
    )
    patch_exact(
        pipeline_path,
        '        first_transformer.pre_cache(X_test, "test")',
        '        if X_test is not None:\n            first_transformer.pre_cache(X_test, "test")',
    )
patch_exact(
    REPO_DIR / "experiment" / "diffprep_experiment.py",
    "prep_pipeline.init_parameters(X_train, X_val, X_test)",
    "prep_pipeline.init_parameters(X_train, X_val, None)",
)
patch_exact(
    REPO_DIR / "experiment" / "diffprep_experiment.py",
    "result, best_model = diff_prep.fit(X_train, y_train, X_val, y_val, X_test, y_test)",
    "result, best_model = diff_prep.fit(X_train, y_train, X_val, y_val, None, None)",
)
patch_exact(
    REPO_DIR / "trainer" / "diffprep_trainer.py",
    "            test_loss, test_acc = self.evaluate(X_test, y_test, X_type='test', max_only=False)",
    "            if X_test is None or y_test is None:\n                test_loss, test_acc = float('nan'), float('nan')\n            else:\n                test_loss, test_acc = self.evaluate(X_test, y_test, X_type='test', max_only=False)",
)
patch_exact(
    REPO_DIR / "extract_and_save_pipeline.py",
    "prep_pipeline.init_parameters(X_train, X_val, X_test)",
    "prep_pipeline.init_parameters(X_train, X_val, None)",
)
patch_exact(
    REPO_DIR / "extract_and_save_pipeline.py",
    "'original_test_acc': result['best_test_acc'],",
    "'original_test_acc': None,",
)
print("Patched DiffPrep: no outer-test access during pipeline search")
os.chdir(REPO_DIR)
print("DiffPrep commit:", commit)
print("SolutionRecommendation commit:", solution_commit)


In [ ]:
# Exact 30-dataset ACORec test corpus supplied for the experiment.
DATASETS = [
    {"dataset_id": 1066, "name": "kc1-binary"},
    {"dataset_id": 1047, "name": "usp05"},
    {"dataset_id": 862, "name": "sleuth-ex2016"},
    {"dataset_id": 40663, "name": "calendarDOW"},
    {"dataset_id": 1054, "name": "mc2"},
    {"dataset_id": 876, "name": "fri-c1"},
    {"dataset_id": 18, "name": "mfeat-morphological"},
    {"dataset_id": 1520, "name": "robot-failures-lp5"},
    {"dataset_id": 1548, "name": "autoUniv-au4"},
    {"dataset_id": 378, "name": "ipums-la-99"},
    {"dataset_id": 1485, "name": "madelon"},
    {"dataset_id": 14, "name": "mfeat-fourier"},
    {"dataset_id": 27, "name": "colic"},
    {"dataset_id": 44956, "name": "abalone", "dataset_key": "abalone"},
    {"dataset_id": 1037, "name": "ada_prior", "dataset_key": "ada_prior"},
    {"dataset_id": 42932, "name": "avila", "dataset_key": "avila"},
    {"dataset_id": 40668, "name": "connect-4", "dataset_key": "connect-4"},
    {"dataset_id": 1471, "name": "eeg", "dataset_key": "eeg"},
    {"dataset_id": 100000, "name": "google", "dataset_key": "google", "source": "kaggle_csv"},
    {"dataset_id": 42165, "name": "house", "dataset_key": "house_prices"},
    {"dataset_id": 41001, "name": "jungle_chess", "dataset_key": "jungle_chess_2pcs_raw_endgame_complete"},
    {"dataset_id": 41671, "name": "micro", "dataset_key": "microaggregation2"},
    {"dataset_id": 1046, "name": "mozilla4", "dataset_key": "mozilla4"},
    {"dataset_id": 46597, "name": "obesity", "dataset_key": "obesity"},
    {"dataset_id": 30, "name": "page-blocks", "dataset_key": "page-blocks"},
    {"dataset_id": 802, "name": "pbcseq", "dataset_key": "pbcseq"},
    {"dataset_id": 722, "name": "pol", "dataset_key": "pol"},
    {"dataset_id": 40922, "name": "run_or_walk", "dataset_key": "Run_or_walk_information"},
    {"dataset_id": 1119, "name": "uscensus", "dataset_key": "USCensus"},
    {"dataset_id": 1497, "name": "wall-robot-nav", "dataset_key": "wall-robot-navigation"},
]

TARGET_OVERRIDES = {42932: "10", 100000: "Rating>4.2"}
IGNORE_OVERRIDES = {42932: ["train", "test"]}

positions = np.array_split(np.arange(len(DATASETS)), NUM_DATASET_SHARDS)
SHARD_DATASETS = [DATASETS[int(i)] for i in positions[DATASET_SHARD_INDEX]]
print(
    f"Shard {DATASET_SHARD_INDEX}/{NUM_DATASET_SHARDS - 1}: "
    f"{len(SHARD_DATASETS)} datasets"
)
display(pd.DataFrame(SHARD_DATASETS))


In [ ]:
def materialize_for_diffprep(spec):
    dataset_key = spec.get("dataset_key", str(spec["dataset_id"]))
    dataset_dir = REPO_DIR / "data" / dataset_key
    data_path = dataset_dir / "data.csv"
    info_path = dataset_dir / "info.json"

    # Google is synthetic (100000), so seed the canonical loader with the
    # exact frozen DiffPrep CSV when it is not attached as a Kaggle input.
    if spec.get("source") == "kaggle_csv":
        canonical_csv = CACHE_DIR / f"{int(spec['dataset_id'])}.csv"
        source_google = REPO_DIR / "data" / dataset_key / "data.csv"
        if not canonical_csv.exists() and source_google.exists():
            shutil.copyfile(source_google, canonical_csv)

    dataset = load_gitlab_openml_dataset(
        int(spec["dataset_id"]),
        cache_dir=str(CACHE_DIR),
        test_dataset_ids=[int(value) for value in EVAL_IDS],
        verbose=True,
        max_samples_if_test=MAX_SAMPLES,
    )
    if dataset is None:
        raise RuntimeError(f"Canonical loader could not load {spec['name']}")

    frame = pd.DataFrame(dataset["X"]).copy()
    frame["target"] = pd.Series(dataset["y"]).reset_index(drop=True)
    if len(frame) < 20 or frame.shape[1] < 2:
        raise ValueError(f"Insufficient usable data: {frame.shape}")

    # DiffPrep's build_data() label-encodes the target. The canonical
    # loader already emits contiguous integer labels, so this is idempotent.
    dataset_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(data_path, index=False)
    info = {
        "label": "target",
        "dataset_id": int(spec["dataset_id"]),
        "dataset_name": spec["name"],
        "source": "canonical_acorec_loader",
        "original_rows": int(dataset.get("original_rows", len(frame))),
        "used_rows": int(len(frame)),
        "raw_features": int(frame.shape[1] - 1),
    }
    info_path.write_text(json.dumps(info, indent=2), encoding="utf-8")
    print(f"Canonical DiffPrep input {spec['name']}: {frame.shape}")
    del dataset, frame
    gc.collect()
    return dataset_key, info


In [ ]:
def as_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def load_saved_pipeline(dataset_key):
    directory = REPO_DIR / "saved_pipelines" / METHOD / dataset_key
    with (directory / "pipeline.pkl").open("rb") as handle:
        pipeline = pickle.load(handle)
    with (directory / "data_splits.pkl").open("rb") as handle:
        split = pickle.load(handle)
    metadata = json.loads((directory / "metadata.json").read_text(encoding="utf-8"))
    return pipeline, split, metadata, directory

def transform_with_diffprep(pipeline, split):
    if not pipeline.is_fitted:
        pipeline.fit(split["X_train"])
    # Test rows are cached only now, after the DiffPrep search is over.
    if "test" not in pipeline.pipeline[0].cache:
        pipeline.pipeline[0].pre_cache(split["X_test"], "test")
    transformed = {}
    with torch.no_grad():
        for part in ("train", "val", "test"):
            value = pipeline.transform(split[f"X_{part}"], X_type=part, max_only=True, resample=False)
            array = as_numpy(value).astype(np.float32, copy=False)
            if not np.isfinite(array).all():
                raise ValueError(f"DiffPrep produced NaN/inf in transformed {part}")
            transformed[part] = array
    return transformed

def assert_split_alignment(raw_split, diffprep_split):
    # DiffPrep globally label-encodes targets; canonical labels can be
    # numeric but non-contiguous (for example Abalone ages).  Verify the
    # row-order witness through a one-to-one raw-label -> DiffPrep-label
    # mapping instead of comparing arbitrary integer code values.
    for part in ("train", "val", "test"):
        raw_y = np.asarray(raw_split[f"y_{part}"]).reshape(-1)
        diff_y = as_numpy(diffprep_split[f"y_{part}"]).reshape(-1)
        if raw_y.shape != diff_y.shape:
            raise RuntimeError(
                f"DiffPrep split mismatch in {part}: "
                f"canonical={raw_y.shape}, diffprep={diff_y.shape}"
            )
        raw_to_diff, diff_to_raw = {}, {}
        for raw_label, diff_label in zip(raw_y, diff_y):
            raw_key, diff_key = str(raw_label), str(diff_label)
            if (
                (raw_key in raw_to_diff and raw_to_diff[raw_key] != diff_key)
                or (diff_key in diff_to_raw and diff_to_raw[diff_key] != raw_key)
            ):
                raise RuntimeError(
                    f"DiffPrep split mismatch in {part}: labels are not "
                    "a consistent one-to-one encoding of the canonical row order"
                )
            raw_to_diff[raw_key] = diff_key
            diff_to_raw[diff_key] = raw_key

def build_raw_split(dataset_key, data_info):
    # Create the same deterministic raw 60/20/20 split as the baseline.
    frame = pd.read_csv(REPO_DIR / "data" / dataset_key / "data.csv")
    target = str(data_info["label"])
    frame = frame.loc[~frame[target].isna()].reset_index(drop=True)
    y = frame.pop(target).reset_index(drop=True)
    n_val = int(len(y) * 0.20)
    n_test = int(len(y) * 0.20)
    indices = np.random.RandomState(SPLIT_SEED).permutation(len(y))
    test_idx = indices[:n_test]
    val_idx = indices[n_test:n_test + n_val]
    train_idx = indices[n_test + n_val:]
    return {
        "X_train": frame.iloc[train_idx].reset_index(drop=True),
        "y_train": y.iloc[train_idx].reset_index(drop=True),
        "X_val": frame.iloc[val_idx].reset_index(drop=True),
        "y_val": y.iloc[val_idx].reset_index(drop=True),
        "X_test": frame.iloc[test_idx].reset_index(drop=True),
        "y_test": y.iloc[test_idx].reset_index(drop=True),
    }

def evaluate_setting(setting, spec, dataset_key, split, data_info, transformed=None, metadata=None):
    X_train = transformed["train"] if transformed is not None else as_numpy(split["X_train"])
    X_val = transformed["val"] if transformed is not None else as_numpy(split["X_val"])
    X_test = transformed["test"] if transformed is not None else as_numpy(split["X_test"])
    y_train = as_numpy(split["y_train"]).ravel()
    y_val = as_numpy(split["y_val"]).ravel()
    y_test = as_numpy(split["y_test"]).ravel()
    started = time.time()
    result, model = evaluate_h2o_frames(
        X_train, y_train, X_val, y_val, X_test, y_test,
        task_type="classification", h2o_preprocessing=None,
        max_runtime_secs=H2O_MAX_RUNTIME_SECS,
        max_runtime_secs_per_model=H2O_MAX_RUNTIME_SECS_PER_MODEL,
        nfolds=H2O_NFOLDS, seed=TRAIN_SEED,
        nthreads=H2O_NTHREADS, max_mem_size=H2O_MAX_MEM_SIZE,
    )
    config = {}
    if metadata is not None:
        config_path = (REPO_DIR / "saved_pipelines" / METHOD / dataset_key / "pipeline_config.json")
        if config_path.exists():
            config = json.loads(config_path.read_text(encoding="utf-8"))
    result.update({
        "dataset_id": int(spec["dataset_id"]), "dataset": spec["name"],
        "dataset_key": dataset_key, "setting": setting,
        "method": METHOD,
        "source": data_info.get("source", "diffprep_fork_snapshot"),
        "original_rows": data_info.get("original_rows"),
        "used_rows": int(len(y_train) + len(y_val) + len(y_test)),
        "raw_features": int(split["X_train"].shape[1]),
        "transformed_features": int(X_train.shape[1]),
        "validation_rows": int(len(y_val)), "split_seed": SPLIT_SEED,
        "train_seed": TRAIN_SEED, "diffprep_commit": commit,
        "diffprep_test_seen_during_search": False,
        "solution_commit": solution_commit, "total_seconds": float(time.time() - started),
        "diffprep_internal_test_accuracy": None if metadata is None else metadata.get("original_test_acc"),
        "diffprep_pipeline_config": json.dumps(config, separators=(",", ":")),
    })
    del model
    gc.collect()
    return result


In [ ]:
# A dedicated output avoids mixing this method's rows with the baseline.
RESULT_PATH = OUTPUT_DIR / f"diffprep_h2o_shard_{DATASET_SHARD_INDEX:02d}_of_{NUM_DATASET_SHARDS:02d}.csv"
def read_rows():
    return pd.read_csv(RESULT_PATH).to_dict("records") if RESULT_PATH.exists() else []
def upsert(rows, row):
    key = (str(row["dataset_id"]), str(row["setting"]))
    rows[:] = [old for old in rows if (str(old.get("dataset_id")), str(old.get("setting"))) != key]
    rows.append(row)
    pd.DataFrame(rows).to_csv(RESULT_PATH, index=False)

rows = read_rows()
completed = {(str(row.get("dataset_id")), str(row.get("setting"))) for row in rows if row.get("status") == "ok"}
positions = np.array_split(np.arange(len(DATASETS)), NUM_DATASET_SHARDS)
SHARD_DATASETS = [DATASETS[int(i)] for i in positions[DATASET_SHARD_INDEX]]
RUN_DATASETS = SHARD_DATASETS[:1] if RUN_MODE == "smoke" else SHARD_DATASETS
for position, spec in enumerate(RUN_DATASETS, start=1):
    dataset_key = spec.get("dataset_key", str(spec["dataset_id"]))
    try:
        dataset_key, data_info = materialize_for_diffprep(spec)
        subprocess.run([sys.executable, "main.py", "--dataset", dataset_key, "--method", METHOD, "--model", "log", "--split_seed", str(SPLIT_SEED), "--train_seed", str(TRAIN_SEED)], cwd=REPO_DIR, check=True)
        subprocess.run([sys.executable, "extract_and_save_pipeline.py", "--dataset", dataset_key, "--method", METHOD, "--split_seed", str(SPLIT_SEED)], cwd=REPO_DIR, check=True)
        subprocess.run([sys.executable, "extract_pipeline_config.py", "--dataset", dataset_key, "--method", METHOD], cwd=REPO_DIR, check=True)
        pipeline, split, metadata, _directory = load_saved_pipeline(dataset_key)
        transformed = transform_with_diffprep(pipeline, split)
        raw_split = build_raw_split(dataset_key, data_info)
        assert_split_alignment(raw_split, split)
    setting = "diffprep"
    if (str(spec["dataset_id"]), setting) in completed:
        print(f"SKIP successful: {spec['name']} / {setting}")
        continue
    print(f"[{position}/{len(RUN_DATASETS)}] {spec['name']} / {setting}")
    row = evaluate_setting(
        setting, spec, dataset_key, split, data_info, transformed, metadata
    )
    upsert(rows, row)
    print(f"H2O outer-test accuracy (DiffPrep): {row['accuracy']:.6f}")
except Exception as error:
    traceback.print_exc()
    if (str(spec["dataset_id"]), "diffprep") not in completed:
        upsert(rows, {"dataset_id": int(spec["dataset_id"]), "dataset": spec["name"], "setting": "diffprep", "status": "failed", "error_type": type(error).__name__, "error": str(error)[:4000]})
    finally:
        gc.collect()
print("Saved:", RESULT_PATH)
display(pd.DataFrame(rows).sort_values(["dataset_id", "setting"]))


## Outputs

Download `diffprep_h2o_shard_XX_of_05.csv` from the Kaggle output directory
and concatenate the five shards. Every row has `setting="diffprep"`.
